<a href="https://colab.research.google.com/github/katzcodes/yolo-vehicle-counter/blob/main/vehicle_counter_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q ultralytics huggingface_hub pandas

In [13]:
import cv2
import numpy as np
import pandas as pd
import os
from base64 import b64encode
from IPython.display import HTML, display
from google.colab import drive
from ultralytics import YOLO

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
VIDEO_IN  = "/content/drive/MyDrive/roundabout/input/cctv052x2004080516x01638.avi"
VIDEO_OUT = "/content/drive/MyDrive/roundabout/output/output_raw.mp4"
CSV_OUT   = "/content/drive/MyDrive/roundabout/output/vehicle_counts.csv"
WEB_OUT   = "/content/display_output.mp4"
CONF      = 0.15

In [31]:
# Class mappings & visual styles
merge = {"car": "Car", "bus": "HCV", "truck": "HCV"}
colors = {"Car": (0, 200, 0), "LCV": (255, 165, 0), "HCV": (0, 0, 255)}

# ROI Polygon Points (Corner vertices: 0 to 4)
ROI_POINTS = [(130, 255), (140, 90), (275, 90), (325, 150), (325, 300)]
roi_poly   = np.array(ROI_POINTS, dtype=np.int32)

# Tracked IDs
tracked_ids = {
    "Car": set(),
    "LCV": set(),
    "HCV": set()
}

In [32]:
model = YOLO("yolo26l.pt")
keep_ids = [i for i, n in model.names.items() if n.lower() in merge]

In [33]:
# Read video metadata
cap = cv2.VideoCapture(VIDEO_IN)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
cap.release()

# Video writer initialization
writer = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

In [34]:
# Font and line scaling calculations
fs = max(0.3, (h / 1080) * 0.6)
th_line = 1 if h < 720 else 2

In [35]:
print("Processing video frames & tracking unique counts...")
for r in model.track(
    source=VIDEO_IN,
    stream=True,
    persist=True,
    tracker="bytetrack.yaml",
    classes=keep_ids,
    conf=CONF,
    imgsz=1280,
    agnostic_nms=True,
    verbose=False,
):
    frame = r.orig_img.copy()

    # Draw semi-transparent ROI polygon overlay on the video frame
    overlay = frame.copy()
    cv2.fillPoly(overlay, [roi_poly], (0, 255, 255))
    cv2.addWeighted(overlay, 0.20, frame, 0.80, 0, frame)
    cv2.polylines(frame, [roi_poly], isClosed=True, color=(0, 255, 255), thickness=2)

    # Process detections
    if r.boxes is not None and r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        tids  = r.boxes.id.int().cpu().tolist()
        clss  = r.boxes.cls.int().cpu().tolist()
        confs = r.boxes.conf.cpu().tolist()

        for box, tid, cls, conf in zip(boxes, tids, clss, confs):
            x1, y1, x2, y2 = map(int, box)

            # Contact point: bottom-center of vehicle bounding box
            cx, cy = (x1 + x2) // 2, y2

            # ROI Filter: Check if vehicle contact point is inside polygon
            if cv2.pointPolygonTest(roi_poly, (float(cx), float(cy)), False) < 0:
                continue

            label = merge[model.names[cls].lower()]
            color = colors.get(label, (255, 255, 255))
            text  = f"{label} #{tid} {conf*100:.0f}%"

            # Log unique ID into corresponding class set
            tracked_ids[label].add(tid)

            # Draw vehicle bounding box & tracking anchor point
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, th_line)
            cv2.circle(frame, (cx, cy), 3, (0, 255, 255), -1)

            # Render text label background
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, 1)
            bg_y1 = max(0, y1 - th - 4)
            cv2.rectangle(frame, (x1, bg_y1), (x1 + tw + 4, y1), color, -1)
            cv2.putText(frame, text, (x1 + 2, max(y1 - 3, th)),
                        cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), 1)

    # ---- Display Live Unique Counter HUD ----
    counter_text = f"Car: {len(tracked_ids['Car'])} | LCV: {len(tracked_ids['LCV'])} | HCV: {len(tracked_ids['HCV'])}"
    cv2.rectangle(frame, (10, 10), (240, 35), (0, 0, 0), -1)
    cv2.putText(frame, counter_text, (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    writer.write(frame)

writer.release()
print("Tracking complete! Saved to Drive:", VIDEO_OUT)

Processing video frames & tracking unique counts...
Tracking complete! Saved to Drive: /content/drive/MyDrive/roundabout/output/output_raw.mp4


In [36]:
df_counts = pd.DataFrame([{
    "Car": len(tracked_ids["Car"]),
    "LCV": len(tracked_ids["LCV"]),
    "HCV": len(tracked_ids["HCV"])
}])

# Ensure parent output directory exists
os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)
df_counts.to_csv(CSV_OUT, index=False)

print("\n--- Final Unique Counts ---")
print(df_counts.to_string(index=False))
print(f"\nSaved CSV counts to: {CSV_OUT}")


--- Final Unique Counts ---
 Car  LCV  HCV
  28    0   21

Saved CSV counts to: /content/drive/MyDrive/roundabout/output/vehicle_counts.csv


In [37]:
print("\nEncoding video for inline Colab playback...")
!ffmpeg -y -i "{VIDEO_OUT}" -vcodec libx264 "{WEB_OUT}" -loglevel quiet

mp4 = open(WEB_OUT, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

display(HTML(f'''
<video width=640 controls autoplay loop muted>
      <source src="{data_url}" type="video/mp4">
</video>
'''))


Encoding video for inline Colab playback...
